# Bronze Layer Exploration — OpenSky Flight Data

This notebook explores the raw flight data ingested from the OpenSky Network into the bronze streaming table `workspace.default.ingest_flights`.

**Purpose:**
- Understand the schema and data types
- Identify null patterns and data quality issues
- Observe value distributions for key columns
- Inform silver-layer cleaning decisions

This notebook is **not** a pipeline source file — it is for ad-hoc exploration only.

---

In [0]:
# Load the bronze table as a static DataFrame for exploration.
# We use spark.read (batch) rather than spark.readStream because this is
# ad-hoc analysis, not a streaming transformation.
df = spark.read.table("workspace.default.ingest_flights")
print(f"Total rows: {df.count()}")

In [0]:
# Print the full schema to understand column names and types.
# This informs which columns need type casting or cleaning in silver.
print("=== Schema ===")
df.printSchema()

In [0]:
# Display 10 sample rows to see actual values.
# Look for: whitespace in strings, null patterns, coordinate ranges.
display(df.limit(10))

In [0]:
# Count nulls per column to identify which fields need cleaning.
# Columns with high null rates may need special handling in silver.
from pyspark.sql.functions import col, count, when, isnull

null_counts = df.select([
    count(when(isnull(c), c)).alias(c) for c in df.columns
])
display(null_counts)

In [0]:
# Check velocity distribution — look for negative values (invalid)
# and understand the typical range.
from pyspark.sql.functions import min, max, avg

df.select(
    min("velocity").alias("min_velocity"),
    max("velocity").alias("max_velocity"),
    avg("velocity").alias("avg_velocity"),
    count(when(col("velocity") < 0, True)).alias("negative_count"),
    count(when(isnull("velocity"), True)).alias("null_count")
).show()

In [0]:
# Validate that latitude and longitude are within expected ranges.
# Latitude: [-90, 90], Longitude: [-180, 180]
from pyspark.sql.functions import col

bad_coords = df.filter(
    (col("latitude") < -90) | (col("latitude") > 90) |
    (col("longitude") < -180) | (col("longitude") > 180)
).count()
print(f"Rows with out-of-range coordinates: {bad_coords}")

# Also check how many rows have null coordinates
null_coords = df.filter(isnull("latitude") | isnull("longitude")).count()
print(f"Rows with null coordinates: {null_coords}")

In [0]:
# Check for potential duplicates based on icao24 + time_position.
# This informs the silver-layer deduplication strategy.
from pyspark.sql.functions import count as count_func

dupes = df.groupBy("icao24", "time_position").agg(
    count_func("*").alias("cnt")
).filter("cnt > 1").orderBy(col("cnt").desc())

print(f"Duplicate (icao24, time_position) combinations: {dupes.count()}")
display(dupes.limit(10))

In [0]:
# Count distinct aircraft (icao24 transponder codes).
# This is one of the key metrics in the gold layer.
distinct_aircraft = df.select("icao24").distinct().count()
print(f"Distinct aircraft (icao24): {distinct_aircraft}")

# Also check for null icao24 — these rows would be dropped in silver.
null_icao = df.filter(isnull("icao24")).count()
print(f"Rows with null icao24: {null_icao}")

In [0]:
# Check if callsign values have leading/trailing whitespace.
# OpenSky data often pads callsigns with spaces.
from pyspark.sql.functions import length, trim, col

sample_callsigns = df.select("callsign").distinct().limit(20)
display(sample_callsigns)

# Compare raw vs trimmed lengths
len_diff = df.filter(
    (length("callsign") != length(trim(col("callsign")))) &
    (col("callsign").isNotNull())
).count()
print(f"Callsigns with extra whitespace: {len_diff}")

## Exploration Summary & Silver Layer Recommendations

Based on the above exploration:

1. **Trim whitespace** on `callsign` and `origin_country` — OpenSky pads these with trailing spaces.
2. **Drop null icao24** — the transponder code is the primary key for aircraft identification.
3. **Filter negative velocity** — physically impossible; keep nulls (may indicate ground status).
4. **Validate coordinates** — drop rows with lat/long outside [-90, 90] / [-180, 180].
5. **Deduplicate** on (icao24, time_position) within a 10-minute watermark window.
6. **Add processing timestamp** `_silver_processed_at` for lineage tracking.

These transformations are implemented in `transformations/silver/cleaned_flights.py`.